In [1]:
import pandas as pd

base = "../../../dataset/"

def load(path):
    # dtype=str keeps IDs as text; keep_default_na=False keeps empty cells as "" instead of NaN
    return pd.read_csv(base + path, sep="\t", dtype=str, keep_default_na=False)

s1 = load("train/train_source1.tsv")
s2 = load("train/train_source2.tsv")
s3 = load("train/train_source3.tsv")
gt = load("train/train_ground_truth.tsv")
t1 = load("test/test_source1.tsv")
t2 = load("test/test_source2.tsv")
t3 = load("test/test_source3.tsv")
print("loaded")

loaded


In [2]:
for name, df in [("train S1", s1), ("train S2", s2), ("train S3", s3),
                 ("test S1", t1), ("test S2", t2), ("test S3", t3)]:
    print(f"{name}: {len(df):,} rows | countries: {df['country'].value_counts().to_dict()}")
print("ground truth rows:", len(gt))

train S1: 2,206,821 rows | countries: {'US': 1323633, 'India': 883188}
train S2: 5,034,616 rows | countries: {'US': 3016817, 'India': 2017799}
train S3: 5,285,603 rows | countries: {'US': 3170056, 'India': 2115547}
test S1: 1,732,544 rows | countries: {'India': 809986, 'US': 663106, 'France': 259452}
test S2: 4,887,273 rows | countries: {'India': 2312565, 'US': 1871330, 'France': 703378}
test S3: 5,082,316 rows | countries: {'India': 2405000, 'US': 1945701, 'France': 731615}
ground truth rows: 2206821


In [3]:
gt["ids"] = gt["matched_entity_ids"].apply(lambda x: [i for i in x.split(",") if i])
gt["n"] = gt["ids"].apply(len)

print("Singleton fraction (your 'do-nothing' score floor):", round((gt["n"] == 0).mean(), 3))
print("\nHow many matches per S1 entity:")
print(gt["n"].value_counts().sort_index().head(15))

pairs = gt.explode("ids").dropna(subset=["ids"])
print("\nMatches from S2 vs S3:", pairs["ids"].str[:2].value_counts().to_dict())

# Does any S2/S3 record get matched to more than one S1? (should be ~0)
print("S2/S3 IDs matched to >1 S1 entity:", (pairs["ids"].value_counts() > 1).sum())

# How many S2/S3 records match nothing at all?
matched = set(pairs["ids"])
print("Unmatched S2 fraction:", round(1 - s2["entity_id"].isin(matched).mean(), 3))
print("Unmatched S3 fraction:", round(1 - s3["entity_id"].isin(matched).mean(), 3))

Singleton fraction (your 'do-nothing' score floor): 0.056

How many matches per S1 entity:
n
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37
Name: count, dtype: int64

Matches from S2 vs S3: {'S3': 3944746, 'S2': 3693619}
S2/S3 IDs matched to >1 S1 entity: 0
Unmatched S2 fraction: 0.266
Unmatched S3 fraction: 0.254


In [4]:
lookup = pd.concat([s1, s2, s3]).set_index("entity_id")
sample = pairs.sample(25, random_state=0)

for _, row in sample.iterrows():
    a, b = lookup.loc[row["source1_entity_id"]], lookup.loc[row["ids"]]
    print(f"[{a['country']}] {a['business_name']}  |  {a['business_address']}")
    print(f"   -> {b['business_name']}  |  {b['business_address']}\n")

[US] 1444 Third Lane Company  |  110 Azalea Avenue, Hueytown, AL
   -> 1444 Thbdr C0mpany Lane  |  110 1/2 Azalea Ave, Hueytown, Alabama

[India] Apex Services  |  Delhi North East, E-18/59 G/F Kh No.207, North West Delhi, E-Block Gali No.18 20Ft, Delhi
   -> Apex Services LLP  |  #298 E-18/59 G/f Kh No.207, Delhi North East, null, दिल्ली

[US] Bailey Currencyshares LLC  |  3824 Moody Parkway, Moody, AL
   -> Bailey Currencyshares  |  Moody Pkwy, Moody, Alabama

[US] Powell, Dunn & Propst Doubleline  |  1708 Landmark Drive, Unit F, Forest Hill, MD
   -> Doubleline Dunn & Propst Powell,  |  Landmark Drive, # F, Forest Hill, Maryland

[India] Ziranth Hotels Private Limited  |  20, First Floor, Super Plaza, Sandesh Press Road, P.O. Bodakdev, Vastrapur, Ahmedabad, Gujarat
   -> ziranth hotels private limited  |  20, Vastrapur, Ahmedabad, ગુજરાત

[India] Su Security LLP  |  Ff 101 102 Shreeji Jyot, Nr Gunatit Jyot, Anand, Gujarat
   -> SU-SECURITY L.L.P.  |  FF 101 102 SHREEJI JYOT, ANAND, 

In [5]:
import sys, numpy as np
sys.path.insert(0, "../src")   # adjust if your notebook isn't in a subfolder next to src/

FRAC = 0.10
rng = np.random.default_rng(42)
s1_tr = s1.sample(frac=FRAC, random_state=42).reset_index(drop=True)
true_tr = pairs[pairs["source1_entity_id"].isin(s1_tr["entity_id"])]

idx_all = pd.concat([s2, s3], ignore_index=True)
keep = idx_all["entity_id"].isin(true_tr["ids"]) | \
       (~idx_all["entity_id"].isin(matched) & (rng.random(len(idx_all)) < FRAC))
idx_tr = idx_all[keep].reset_index(drop=True)
print(len(s1_tr), "S1 |", len(idx_tr), "S2+S3 |", len(true_tr), "true pairs")

220682 S1 | 1031762 S2+S3 | 763411 true pairs


In [6]:
from normalize import add_normalized
s1_tr = add_normalized(s1_tr, n_jobs=4)
idx_tr = add_normalized(idx_tr, n_jobs=4)
s1_tr.to_pickle("s1_tr.pkl"); idx_tr.to_pickle("idx_tr.pkl")
s1_tr[["business_name", "name_n", "addr_n"]].head()

,business_name,name_n,addr_n
0,Pediatric Medicine PLLC,pediatric medicine,4850 20 otisco ny
1,Fetech National Twin,fetech national twin,19034 woodburn road woodburn in
2,General Design Innovations LLC,general design innovations,7241 osage avenue mesa az
3,Construction Ideaz Papers Private Limited,construction ideaz papers,building 4/606 prabhul cottage karimbalur elak...
4,Penaloza and Bittle First Inc.,penaloza bittle first,4255 charleswood avenue memphis tn


In [7]:
from blocking import block, recall_at_k
cand = block(s1_tr, idx_tr, k=30)
recall_at_k(cand, s1_tr, idx_tr, true_tr)

  US: 132,631 S1 vs 619,709 S2/S3, vocab 257,623, 77s
  India: 88,051 S1 vs 412,053 S2/S3, vocab 268,472, 67s


,k,recall,cands_per_s1
0,5,0.875510,4.998926
1,10,0.938956,9.995210
2,20,0.954033,19.979835
3,30,0.960500,29.953036


In [8]:
ix = pd.Series(np.arange(len(idx_tr)), index=idx_tr["entity_id"])
s1p = pd.Series(np.arange(len(s1_tr)), index=s1_tr["entity_id"])
found = set(zip(cand["s1_pos"], cand["cand_pos"]))
missed = [(a, b) for a, b in zip(s1p[true_tr["source1_entity_id"]], ix[true_tr["ids"]])
          if (a, b) not in found]
for a, b in missed[:15]:
    print(s1_tr.loc[a, "block_text"], "\n   ->", idx_tr.loc[b, "block_text"], "\n")

new india floor 401 tg sri niketan abhinandana aprt ayyappa society near cgr int place 250 251 madhapur hyderabad rangareddy 
   -> new services  

dynamic biomedical networks 270 school hill road cecil township pa 
   -> dynamicbiomedicalnetworks com 27 school hill road canonsburg pa 

global consultants mumbai mh mumbai city 306 sangeet plaza opposite marol fire brigade marol maroshi road marol andheri east 
   -> globl kmsltemts praivet 306/2 mumbai city mumbai mharastr 306 

hari consultancy flate 406 shakar splun vamali sama savli road vadodara gj 
   -> hari consultancy  

physical therapy associates group 4303 elkins avenue b nashville tn 
   -> group physical therapy asseatiaes 004303 elkins avenue po box 5980 nashville tn 

board of sanitation 17945 rockwell avenue homewood il 
   -> board of saninttation  

vision industries fazilka road village ghubaya teh jalalabad jalalabad firozpur pb 
   -> vision industries  

united institute vi tx 413 parsons street manor 
   -> unite

In [9]:
del s2, s3, idx_all, lookup, t1, t2, t3, matched, gt
import gc; gc.collect()

20

In [10]:
import importlib, normalize, blocking
importlib.reload(normalize); importlib.reload(blocking)
from normalize import add_normalized
from blocking import block, recall_at_k
s1_tr = add_normalized(s1_tr, n_jobs=4); idx_tr = add_normalized(idx_tr, n_jobs=4)
cand = block(s1_tr, idx_tr, k=20)
recall_at_k(cand, s1_tr, idx_tr, true_tr, ks=[10, 20])

  US: 132,631 S1 vs 619,709 S2/S3, vocab 248,378, 111s
  India: 88,051 S1 vs 412,053 S2/S3, vocab 267,700, 75s


,k,recall,cands_per_s1
0,10,0.941793,9.995210
1,20,0.956202,19.979722


In [11]:
from features import make_features, FEATURE_COLS, assign, f05_macro
feat = make_features(cand, s1_tr, idx_tr, n_jobs=4)

s1p = pd.Series(np.arange(len(s1_tr)), index=s1_tr["entity_id"])
ixp = pd.Series(np.arange(len(idx_tr)), index=idx_tr["entity_id"])
truth = pd.DataFrame({"s1_pos": s1p[true_tr["source1_entity_id"]].to_numpy(),
                      "cand_pos": ixp[true_tr["ids"]].to_numpy()})
tkey = set(zip(truth["s1_pos"], truth["cand_pos"]))
feat["label"] = [k in tkey for k in zip(feat["s1_pos"], feat["cand_pos"])]
is_val = feat["s1_pos"] % 5 == 0          # 20% of S1 entities held out
print(len(feat), "pairs |", feat["label"].mean().round(3), "are true matches")

4409165 pairs | 0.166 are true matches


In [12]:
import lightgbm as lgb
params = dict(objective="binary", learning_rate=0.1, num_leaves=63,
              min_child_samples=50, feature_fraction=0.8, verbose=-1)
dtr = lgb.Dataset(feat.loc[~is_val, FEATURE_COLS], feat.loc[~is_val, "label"])
dva = lgb.Dataset(feat.loc[is_val, FEATURE_COLS], feat.loc[is_val, "label"])
model = lgb.train(params, dtr, 1000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
model.save_model("lgbm_v1.txt")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0172103
Early stopping, best iteration is:
[106]	valid_0's binary_logloss: 0.0171264


In [13]:
val = feat[is_val].copy()
val["prob"] = model.predict(val[FEATURE_COLS])
val_s1 = np.flatnonzero(np.arange(len(s1_tr)) % 5 == 0)
for th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    print(th, round(f05_macro(assign(val, th), truth, val_s1), 4))

0.3 0.955
0.4 0.9581
0.5 0.9602
0.6 0.9611
0.7 0.9617
0.8 0.9612
